# Donut Training Diagnostics

Loads a checkpoint (or the base model) and inspects what goes in and what comes out.

In [ ]:
import sys

sys.path.insert(0, ".")

from pathlib import Path
import torch
import numpy as np
from PIL import Image
from transformers import DonutProcessor

from label_formatter import LabelFormatter
from model_module import DonutModule
from dataset import DonutDataset, load_local_samples

# ── Config ──────────────────────────────────────────────────────────────────
MODEL_NAME = "naver-clova-ix/donut-base"
CKPT_PATH = (
    None  # e.g. 'lightning_logs/version_0/checkpoints/epoch=0-val_loss=12.3456.ckpt'
)
DATA_DIR = "../data"  # set to None to use a bare image with no label
FALLBACK_IMAGE = "../test_data/test_data.jpg"
TASK_START_TOKEN = "<s_donut>"
MAX_NEW_TOKENS = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 1. Load model + processor

In [ ]:
additional_tokens = [TASK_START_TOKEN] + LabelFormatter.get_all_tokens()

processor = DonutProcessor.from_pretrained(MODEL_NAME)
processor.tokenizer.add_special_tokens({"additional_special_tokens": additional_tokens})

if CKPT_PATH:
    module = DonutModule.load_from_checkpoint(CKPT_PATH, processor=processor)
    print(f"Loaded checkpoint: {CKPT_PATH}")
else:
    module = DonutModule(
        model_name=MODEL_NAME,
        processor=processor,
        additional_tokens=additional_tokens,
    )
    print("Loaded base pretrained model (no checkpoint)")

model = module.model.to(DEVICE).eval()
tok = processor.tokenizer

print(f"\nmodule.training  : {module.training}")
print(f"model.training   : {model.training}")
print(f"encoder.training : {model.encoder.training}")
print(f"decoder.training : {model.decoder.training}")

## 2. Load a sample (real data or fallback image)

In [ ]:
from IPython.display import display

if DATA_DIR is not None:
    data_path = Path(DATA_DIR)
    samples = load_local_samples(
        images_dir=data_path / "images",
        annotations_dir=data_path / "annotations",
    )
    assert samples, f"No samples found in {DATA_DIR}"
    print(f"Loaded {len(samples)} samples from {DATA_DIR}")
    dataset = DonutDataset(
        data=samples,
        processor=processor,
        label_formatter=LabelFormatter(),
        task_start_token=TASK_START_TOKEN,
        max_target_length=512,
    )
    item = dataset[0]
    pv = item["pixel_values"].unsqueeze(0)  # (1, 3, H, W)
    labels = item["labels"]  # (max_len,)
    raw_image = (
        Image.open(samples[0]["image"])
        if isinstance(samples[0]["image"], (str, Path))
        else samples[0]["image"]
    ).convert("RGB")
    print(f"Sample fields: {[f['field_name'] for f in samples[0]['fields']]}")
else:
    print("DATA_DIR not set — using fallback image with no label")
    raw_image = Image.open(FALLBACK_IMAGE).convert("RGB")
    pv = processor(raw_image, return_tensors="pt").pixel_values
    labels = None

display(raw_image.resize((320, 240)))

## 3. Token ID sanity checks

In [ ]:
start_id = model.config.decoder_start_token_id
pad_id = model.config.pad_token_id
vocab_size = len(tok)

print(f"Vocab size               : {vocab_size}")
print(f'decoder_start_token_id   : {start_id}  → "{tok.decode([start_id])}"')
print(f'pad_token_id             : {pad_id}    → "{tok.decode([pad_id])}"')
print(f'eos_token_id             : {tok.eos_token_id}  → "{tok.eos_token}"')
print(f"Random-chance loss (nats): {np.log(vocab_size):.3f}")
print()

print("Additional token IDs:")
for t in additional_tokens:
    tid = tok.convert_tokens_to_ids(t)
    ok = tid != tok.unk_token_id
    print(f"  {t:40s} → {tid}  {'OK' if ok else 'MISSING (maps to UNK!)'}")

## 4. Pixel value stats

In [ ]:
print(f"pixel_values shape : {tuple(pv.shape)}")
print(f"dtype              : {pv.dtype}")
print(f"min / max / mean   : {pv.min():.4f} / {pv.max():.4f} / {pv.mean():.4f}")
if pv.min() >= -1.1 and pv.max() <= 1.1:
    print("Range looks correct (roughly [-1, 1])")
else:
    print("WARNING: range outside [-1, 1] — check normalization")

## 5. Label decoding round-trip

In [ ]:
if labels is None:
    print("No labels (DATA_DIR not set)")
else:
    non_pad = labels[labels != tok.pad_token_id]
    non_pad_unmasked = labels.clone()
    non_pad_unmasked[non_pad_unmasked == -100] = tok.pad_token_id
    non_pad_ids = non_pad_unmasked[non_pad_unmasked != tok.pad_token_id]

    print(f"Non-pad label tokens : {len(non_pad_ids)}")
    print(f"Decoded              : {tok.decode(non_pad_ids)}")
    print()
    ends_with_eos = (
        (non_pad_ids[-1].item() == tok.eos_token_id) if len(non_pad_ids) > 0 else False
    )
    print(
        f"Ends with EOS token  : {ends_with_eos} {'✓' if ends_with_eos else '✗ labels missing EOS!'}"
    )

## 6. Teacher-forced forward pass

In [ ]:
if labels is None:
    print("No labels — skipping teacher-forced pass")
else:
    pv_dev = pv.to(DEVICE)
    lbl_dev = labels.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        out = model(pixel_values=pv_dev, labels=lbl_dev)

    print(f"Teacher-forced loss: {out.loss.item():.4f}")
    print(f"(Random-chance:      {np.log(vocab_size):.3f})")
    print()

    logits = out.logits[0]
    pred_ids = logits.argmax(dim=-1)
    true_ids = labels[: logits.shape[0]]

    active = true_ids != -100
    pred_active = pred_ids[active]
    true_active = true_ids[active]

    print("Pos  | True token              | Predicted")
    print("-" * 62)
    for i, (t, p) in enumerate(zip(true_active.tolist(), pred_active.tolist())):
        tt = tok.decode([t])
        pt = tok.decode([p])
        match = "✓" if t == p else "✗"
        print(f"{i:4d} | {tt:24s} | {pt:24s} {match}")
        if i >= 30:
            print(f"     ... ({len(true_active) - 31} more positions)")
            break

## 7. Greedy generation (no teacher forcing)

In [ ]:
pv_dev = pv.to(DEVICE)
with torch.no_grad():
    gen_ids = model.generate(
        pixel_values=pv_dev,
        decoder_start_token_id=start_id,
        max_new_tokens=MAX_NEW_TOKENS,
        eos_token_id=tok.eos_token_id,
        pad_token_id=pad_id,
    )

generated = tok.decode(gen_ids[0], skip_special_tokens=False)
print("Generated:")
print(repr(generated))

if labels is not None:
    unmasked = labels.clone()
    unmasked[unmasked == -100] = tok.pad_token_id
    active_ids = unmasked[unmasked != tok.pad_token_id]
    print("\nTarget:")
    print(repr(tok.decode(active_ids)))